# iPro-MP Official Five-Fold Inference: Colab A100

This notebook evaluates the official iPro-MP E. coli model on the same
SeqTrainer validation and held-out test CSVs used by CNN-v2 and DNABERT2.
It performs inference only: there are no epochs and no iPro-MP retraining.

**Required runtime:** NVIDIA A100 GPU. The five official fold checkpoints are
loaded one at a time to stay within A100 memory.

## Fixed scientific contract

- Same GSE144621 predefined split files and labels.
- Seed `42`.
- Official E. coli species ID `10` model.
- Official five-fold ensemble; positive-class probabilities are averaged.
- Overlapping 6-mer tokenization and model length `300`.
- Validation-only MCC threshold selection.
- Held-out test MCC and AUPRC for final comparison.
- Shared SeqTrainer metrics and artifact schema.

The A100 profile uses physical inference batch size `32` and runs validation and
test by default. Train-split predictions are optional because they are not used
for model selection or final reporting. Omitting train inference reduces runtime
without changing validation or test metrics.

## 1. Verify the Colab A100 accelerator


In [ ]:
import subprocess

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu_info)
if "A100" not in gpu_info:
    raise RuntimeError("Select an NVIDIA A100 Colab runtime and rerun from the top.")

## 2. Define reproducible paths and versions


In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
SEQTRAINER_COMMIT = "659cb28a59b44057607329babf16b196b69b1e6f"
REPO_DIR = Path("/content/SeqTrainer")

MINIFORGE_DIR = Path("/content/miniforge3")
ENV_DIR = Path("/content/envs/seqtrainer-ipromp-a100")
ENV_PYTHON = ENV_DIR / "bin" / "python"
ENV_SEQTRAINER = ENV_DIR / "bin" / "seqtrainer"
PIP_CACHE_DIR = Path("/content/pip-cache")

CONFIG_RELATIVE_PATH = Path("notebooks/colab_benchmarks/config/ipromp_external_a100.toml")
LOCAL_DATA_DIR = REPO_DIR / "data" / "promoter_classification"
LOCAL_MODEL_ROOT = Path("/content/ipromp_models")
DNABERT_DIR = LOCAL_MODEL_ROOT / "DNABERT-6"
IPROMP_MODEL_DIR = LOCAL_MODEL_ROOT / "ipromp_ecoli"
RUN_DIR = REPO_DIR / "outputs" / "benchmarks" / "ipromp_external_a100_ep_genomic_order"

print("Pinned SeqTrainer commit:", SEQTRAINER_COMMIT)

## 3. Create the pinned Python 3.10 environment and check out SeqTrainer


In [ ]:
installer = Path("/content/Miniforge3-Linux-x86_64.sh")
if not (MINIFORGE_DIR / "bin" / "conda").exists():
    subprocess.run(
        ["wget", "-q", "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh", "-O", str(installer)],
        check=True,
    )
    subprocess.run(["bash", str(installer), "-b", "-p", str(MINIFORGE_DIR)], check=True)

conda = MINIFORGE_DIR / "bin" / "conda"
if not ENV_PYTHON.exists():
    subprocess.run([str(conda), "create", "-y", "-p", str(ENV_DIR), "python=3.10", "pip"], check=True)

install_env = os.environ.copy()
install_env["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip<25", "setuptools", "wheel"], check=True, env=install_env)
subprocess.run(
    [
        str(ENV_PYTHON), "-m", "pip", "install",
        "numpy==1.24.4", "pandas==2.0.3", "scikit-learn==1.3.2",
        "torch==2.2.2", "transformers==4.29.2", "huggingface_hub<1.0",
        "remotezip", "requests", "packaging", "tomli",
    ],
    check=True,
    env=install_env,
)

if REPO_DIR.exists():
    current = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True)
    if current.returncode != 0 or current.stdout.strip() != SEQTRAINER_COMMIT:
        shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SEQTRAINER_COMMIT], check=True)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR)], check=True, env=install_env)

CONFIG_PATH = REPO_DIR / CONFIG_RELATIVE_PATH
assert CONFIG_PATH.exists(), CONFIG_PATH
print("Config:", CONFIG_PATH)

## 4. Verify the pinned environment


In [ ]:
# Verify pinned iPro-MP environment and GPU.
# Why this cell exists:
# - We run this check inside the pinned iPro-MP virtual environment.
# - SeqTrainer imports rdflib through its SBOL/data modules, so rdflib must exist
#   even if this iPro-MP benchmark is using CSV files.

import json
import subprocess
import textwrap

# Install lightweight SeqTrainer import dependency if missing.
subprocess.run(
    [str(ENV_PYTHON), "-m", "pip", "install", "rdflib"],
    check=True,
)

environment_check = r'''
import json
import torch
import transformers
import seqtrainer

payload = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seqtrainer": seqtrainer.__file__,
}

print(json.dumps(payload, indent=2))

if not payload["cuda_available"]:
    raise SystemExit(
        "CUDA is not available inside the pinned iPro-MP environment. "
        "In Colab, go to Runtime > Change runtime type > GPU, then rerun setup cells."
    )

gpu_name = payload["cuda_device"] or ""
allowed_gpus = ["A100", "A100"]

if not any(name in gpu_name for name in allowed_gpus):
    print(
        "WARNING: This notebook was designed for A100, but Colab assigned: "
        f"{gpu_name}. Continuing because CUDA is available."
    )
'''

result = subprocess.run(
    [str(ENV_PYTHON), "-c", environment_check],
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)

print("\n===== STDOUT =====")
print(result.stdout)

print("\n===== STDERR =====")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Pinned iPro-MP environment check failed. Read STDOUT/STDERR above."
    )

## 5. Mount Drive and choose persistent locations


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive", force_remount=False)

split_file_names = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

search_roots = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path("/content/drive"),
]

def contains_all_splits(folder):
    return folder.is_dir() and all((folder / name).exists() for name in split_file_names.values())

matches = []
for root in search_roots:
    if not root.exists():
        continue
    for train_file in root.rglob(split_file_names["train"]):
        folder = train_file.parent
        if contains_all_splits(folder) and folder not in matches:
            matches.append(folder)

if not matches:
    raise FileNotFoundError(
        "Could not find the benchmark split files in MyDrive, Shared drives, or mounted Drive. "
        "If AIxBio is shared with you, add a shortcut to My Drive first, then rerun this cell."
    )

if len(matches) > 1:
    print("Multiple matching dataset folders found:")
    for i, folder in enumerate(matches, start=1):
        print(f"{i}. {folder}")
    raise RuntimeError("Choose the correct folder above and set DRIVE_DATA_DIR manually.")

DRIVE_DATA_DIR = matches[0]
print("Using dataset folder:", DRIVE_DATA_DIR)

LOCAL_DATA_DIR = REPO_DIR / "data" / "promoter_classification"
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

for split, file_name in split_file_names.items():
    source = DRIVE_DATA_DIR / file_name
    target = LOCAL_DATA_DIR / file_name
    shutil.copy2(source, target)
    print(f"Copied {split}: {source} -> {target}")

## 6. Locate, stage, and audit the shared CSV splits


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import shutil
import pandas as pd

# Mount Google Drive.
drive.mount("/content/drive", force_remount=False)

# These must be defined before DRIVE_OUTPUT_DIR.
MY_DRIVE = Path("/content/drive/MyDrive")
SHARED_DRIVES = Path("/content/drive/Shareddrives")

# Local repo/data locations.
LOCAL_DATA_DIR = REPO_DIR / "data" / "promoter_classification"
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Persistent place in Drive to save notebook audit/output files.
DRIVE_OUTPUT_DIR = MY_DRIVE / "SeqTrainer" / "outputs" / "ipromp_a100_seed42"
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional manual override.
# If automatic search finds duplicate folders, paste the correct folder here.
# Example:
# DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AIxBio/Promoter Classification/Data")
DRIVE_DATA_DIR = None

SPLIT_FILES = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

def contains_all_splits(directory):
    directory = Path(directory)
    return directory.is_dir() and all((directory / name).is_file() for name in SPLIT_FILES.values())

def discover_data_directories():
    if DRIVE_DATA_DIR is not None:
        candidate = Path(DRIVE_DATA_DIR)
        if not contains_all_splits(candidate):
            raise FileNotFoundError(f"DRIVE_DATA_DIR is incomplete: {candidate}")
        return [candidate]

    matches = []

    search_roots = [
        MY_DRIVE / "AIxBio",
        MY_DRIVE / "AI x Bio",
        MY_DRIVE,
        SHARED_DRIVES,
    ]

    for root in search_roots:
        if not root.exists():
            continue

        for train_path in root.rglob(SPLIT_FILES["train"]):
            folder = train_path.parent
            if contains_all_splits(folder):
                resolved = folder.resolve()
                if resolved not in matches:
                    matches.append(resolved)

        if matches and root in {MY_DRIVE / "AIxBio", MY_DRIVE / "AI x Bio"}:
            break

    return matches

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

matches = discover_data_directories()

if not matches:
    raise FileNotFoundError(
        "No Drive directory contains all three shared benchmark CSVs. "
        "If AIxBio is a shared folder, add a shortcut to My Drive or set DRIVE_DATA_DIR manually."
    )

if len(matches) > 1:
    print("Multiple matching dataset folders found:")
    for path in matches:
        print("-", path)
    raise RuntimeError("Set DRIVE_DATA_DIR to the intended directory and rerun this cell.")

selected_data_dir = matches[0]
print("Using dataset folder:", selected_data_dir)

audit = {}

for split, filename in SPLIT_FILES.items():
    source = selected_data_dir / filename
    target = LOCAL_DATA_DIR / filename

    shutil.copy2(source, target)
    frame = pd.read_csv(target)

    missing_columns = {"sequence", "label"}.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{target} is missing required columns: {sorted(missing_columns)}")

    labels = sorted(frame["label"].dropna().unique().tolist())
    if labels != [0, 1]:
        raise ValueError(f"{split} labels must be [0, 1], found {labels}")

    audit[split] = {
        "source": str(source),
        "target": str(target),
        "rows": int(len(frame)),
        "sha256": sha256(target),
        "label_counts": {
            str(key): int(value)
            for key, value in frame["label"].value_counts().sort_index().to_dict().items()
        },
    }

print(json.dumps(audit, indent=2))

audit_path = DRIVE_OUTPUT_DIR / "input_split_audit.json"
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")
print("Saved audit to:", audit_path)

## 7. Download or restore the official model files

The five E. coli fold checkpoints are about 1.8 GB in total. The first run can
therefore take time. When Drive caching is enabled, later sessions copy the
same verified files into fast Colab-local storage before inference.

In [ ]:
from pathlib import Path
import shutil
import subprocess

# Drive/root paths. These make the cell runnable even after a runtime restart.
MY_DRIVE = Path("/content/drive/MyDrive")

# Cache downloaded iPro-MP model 10 folds and DNABERT-6 files in Drive.
# True = first run downloads once, later runs restore from Drive.
# False = download only into the current Colab runtime.
CACHE_MODELS_TO_DRIVE = True

# Local model locations in the current Colab runtime.
LOCAL_MODEL_ROOT = Path("/content/seqtrainer_ipromp_models")
IPROMP_MODEL_DIR = LOCAL_MODEL_ROOT / "ipromp_ecoli"
DNABERT_DIR = LOCAL_MODEL_ROOT / "DNABERT-6"

# Persistent model cache in Drive.
DRIVE_MODEL_CACHE = MY_DRIVE / "SeqTrainer" / "model_cache" / "ipromp_a100"

LOCAL_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
IPROMP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DNABERT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_CACHE.mkdir(parents=True, exist_ok=True)

EXPECTED_FOLDS = [f"10_fold_{fold}.pth" for fold in range(1, 6)]
DNABERT_FILES = ["config.json", "pytorch_model.bin", "vocab.txt"]

def model_cache_complete(root):
    root = Path(root)
    return (
        all((root / "ipromp_ecoli" / name).is_file() for name in EXPECTED_FOLDS)
        and all((root / "DNABERT-6" / name).is_file() for name in DNABERT_FILES)
    )

if CACHE_MODELS_TO_DRIVE and model_cache_complete(DRIVE_MODEL_CACHE):
    print("Restoring model files from Drive cache")
    shutil.copytree(DRIVE_MODEL_CACHE, LOCAL_MODEL_ROOT, dirs_exist_ok=True)
else:
    print("Downloading official iPro-MP E. coli fold weights and DNABERT-6 files")

    downloader = REPO_DIR / "notebooks/benchmarks_sg/ipromp_benchmark/download_ecoli_weights.py"
    subprocess.run(
        [str(ENV_PYTHON), str(downloader), "--output-dir", str(IPROMP_MODEL_DIR)],
        check=True,
    )

    download_dnabert = (
        "from huggingface_hub import snapshot_download; "
        f"snapshot_download(repo_id='zhihan1996/DNA_bert_6', local_dir={str(DNABERT_DIR)!r}, "
        "allow_patterns=['config.json','pytorch_model.bin','special_tokens_map.json','tokenizer_config.json','vocab.txt'])"
    )
    subprocess.run([str(ENV_PYTHON), "-c", download_dnabert], check=True)

    if CACHE_MODELS_TO_DRIVE:
        print("Saving model files to Drive cache")
        shutil.copytree(LOCAL_MODEL_ROOT, DRIVE_MODEL_CACHE, dirs_exist_ok=True)

if not model_cache_complete(LOCAL_MODEL_ROOT):
    raise FileNotFoundError("The required DNABERT-6 or five E. coli fold files are incomplete.")

print("Model files ready in:", LOCAL_MODEL_ROOT)
print("iPro-MP E. coli folds:", IPROMP_MODEL_DIR)
print("DNABERT-6 files:", DNABERT_DIR)

In [ ]:
EXPECTED_FOLDS = [f"10_fold_{fold}.pth" for fold in range(1, 6)]
DNABERT_FILES = ["config.json", "pytorch_model.bin", "vocab.txt"]

def model_cache_complete(root):
    root = Path(root)
    return (
        all((root / "ipromp_ecoli" / name).is_file() for name in EXPECTED_FOLDS)
        and all((root / "DNABERT-6" / name).is_file() for name in DNABERT_FILES)
    )

if CACHE_MODELS_TO_DRIVE and model_cache_complete(DRIVE_MODEL_CACHE):
    print("Restoring model files from Drive cache")
    shutil.copytree(DRIVE_MODEL_CACHE, LOCAL_MODEL_ROOT, dirs_exist_ok=True)
else:
    IPROMP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    DNABERT_DIR.mkdir(parents=True, exist_ok=True)
    downloader = REPO_DIR / "notebooks/benchmarks_sg/ipromp_benchmark/download_ecoli_weights.py"
    subprocess.run([str(ENV_PYTHON), str(downloader), "--output-dir", str(IPROMP_MODEL_DIR)], check=True)
    download_dnabert = (
        "from huggingface_hub import snapshot_download; "
        f"snapshot_download(repo_id='zhihan1996/DNA_bert_6', local_dir={str(DNABERT_DIR)!r}, "
        "allow_patterns=['config.json','pytorch_model.bin','special_tokens_map.json','tokenizer_config.json','vocab.txt'])"
    )
    subprocess.run([str(ENV_PYTHON), "-c", download_dnabert], check=True)
    if CACHE_MODELS_TO_DRIVE:
        DRIVE_MODEL_CACHE.mkdir(parents=True, exist_ok=True)
        shutil.copytree(LOCAL_MODEL_ROOT, DRIVE_MODEL_CACHE, dirs_exist_ok=True)

if not model_cache_complete(LOCAL_MODEL_ROOT):
    raise FileNotFoundError("The required DNABERT-6 or five E. coli fold files are incomplete.")
print("Model files ready in", LOCAL_MODEL_ROOT)

## 8. Assert the A100 iPro-MP scientific contract


In [ ]:
import tomllib

with CONFIG_PATH.open("rb") as handle:
    config = tomllib.load(handle)

assert config["experiment"]["seed"] == 42
assert config["split"]["seed"] == 42
assert config["model"]["params"]["species_id"] == 10
assert config["model"]["params"]["folds"] == 5
assert config["model"]["params"]["kmer_size"] == 6
assert config["model"]["params"]["max_length"] == 300
assert config["model"]["params"]["inference_batch_size"] == 32
assert config["training"]["max_epochs"] == 0
assert config["evaluation"]["threshold_strategy"] == "validation_mcc"
assert config["environment"]["precision"] == "float32"
print("Configuration checks passed")

## 9. Prepare FASTA files with stable IDs


In [ ]:
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
subprocess.run(
    [str(ENV_SEQTRAINER), "benchmark", "prepare-ipromp", str(CONFIG_PATH), "--base-dir", str(REPO_DIR), "--output-dir", str(RUN_DIR)],
    check=True,
)
for split in ("validation", "test"):
    path = RUN_DIR / "ipromp_fasta" / f"{split}.fasta"
    if not path.exists():
        raise FileNotFoundError(path)
print("FASTA preparation complete")

## 10. Run sequential five-fold inference with restart support

Validation runs first because its probabilities define the single MCC threshold.
Each completed split is copied to Drive. If Colab disconnects, rerun from the
top and this cell will restore completed split predictions instead of repeating
them.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

# iPro-MP is inference-only here.
# False = run only validation + test, faster and enough for benchmark comparison.
# True = also run train predictions, useful only if you want full artifact coverage.
RUN_TRAIN_SPLIT = False

splits_to_run = ["validation", "test"]
if RUN_TRAIN_SPLIT:
    splits_to_run.insert(0, "train")

print("Running iPro-MP inference for splits:", splits_to_run)

In [ ]:
PERSISTENT_PREDICTIONS = DRIVE_OUTPUT_DIR / "external_predictions"
LOCAL_PREDICTIONS = RUN_DIR / "external_predictions"
PERSISTENT_PREDICTIONS.mkdir(parents=True, exist_ok=True)
LOCAL_PREDICTIONS.mkdir(parents=True, exist_ok=True)

splits_to_run = ["validation", "test"]
if RUN_TRAIN_SPLIT:
    splits_to_run.insert(0, "train")

for split in splits_to_run:
    local_csv = LOCAL_PREDICTIONS / f"{split}_predictions.csv"
    persistent_csv = PERSISTENT_PREDICTIONS / local_csv.name
    persistent_metadata = persistent_csv.with_suffix(".metadata.json")
    if persistent_csv.exists():
        print(f"Restoring completed {split} predictions from Drive")
        shutil.copy2(persistent_csv, local_csv)
        if persistent_metadata.exists():
            shutil.copy2(persistent_metadata, local_csv.with_suffix(".metadata.json"))
        continue

    command = [
        str(ENV_PYTHON), "-m", "seqtrainer.adapters.ipromp_inference",
        "--input-fasta", str(RUN_DIR / "ipromp_fasta" / f"{split}.fasta"),
        "--output-csv", str(local_csv),
        "--split", split,
        "--dnabert-dir", str(DNABERT_DIR),
        "--model-dir", str(IPROMP_MODEL_DIR),
        "--species-id", "10",
        "--kmer-size", "6",
        "--max-length", "300",
        "--batch-size", "4",
        "--seed", "42",
        "--device", "cuda",
    ]
    print("Running", split, "inference")
    subprocess.run(command, check=True)
    shutil.copy2(local_csv, persistent_csv)
    metadata = local_csv.with_suffix(".metadata.json")
    if metadata.exists():
        shutil.copy2(metadata, persistent_metadata)
    print("Saved", split, "predictions to Drive")

## 11. Evaluate with the shared validation-only threshold policy


In [ ]:
subprocess.run(
    [str(ENV_SEQTRAINER), "benchmark", "run", str(CONFIG_PATH), "--base-dir", str(REPO_DIR), "--output-dir", str(RUN_DIR), "--strict"],
    check=True,
)
shutil.copytree(RUN_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
print("Persistent results:", DRIVE_OUTPUT_DIR)

## 12. Inspect metrics and plots


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve
import matplotlib.pyplot as plt

metrics_path = DRIVE_OUTPUT_DIR / "metrics.csv"
predictions_path = DRIVE_OUTPUT_DIR / "predictions.csv"
manifest_path = DRIVE_OUTPUT_DIR / "manifest.json"
for path in (metrics_path, predictions_path, manifest_path):
    if not path.exists():
        raise FileNotFoundError(path)

metrics = pd.read_csv(metrics_path)
display(metrics)
test_metrics = metrics.loc[metrics["split"] == "test"].iloc[0]
print("Held-out test MCC:", round(float(test_metrics["mcc"]), 6))
print("Held-out test AUPRC:", round(float(test_metrics["auprc"]), 6))
print("Validation-selected threshold:", round(float(test_metrics["threshold"]), 6))

predictions = pd.read_csv(predictions_path)
test = predictions.loc[predictions["split"] == "test"]
y_true = test["label"].to_numpy()
y_score = test["probability"].to_numpy()
y_pred = test["prediction"].to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=axes[0], colorbar=False)
fpr, tpr, _ = roc_curve(y_true, y_score)
axes[1].plot(fpr, tpr, label=f"AUROC={float(test_metrics['auroc']):.3f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="grey")
axes[1].legend()
precision, recall, _ = precision_recall_curve(y_true, y_score)
axes[2].plot(recall, precision, label=f"AUPRC={float(test_metrics['auprc']):.3f}")
axes[2].legend()
plt.tight_layout()
plt.show()

## 13. Verify the inference artifact contract


In [ ]:
required = [
    "metrics.csv", "metrics.json", "predictions.csv", "manifest.json",
    "ipromp_id_mapping.csv", "external_predictions/validation_predictions.csv",
    "external_predictions/test_predictions.csv",
]
missing = [name for name in required if not (DRIVE_OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing artifacts: {missing}")
print("Completed A100 iPro-MP artifacts:")
for path in sorted(DRIVE_OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(DRIVE_OUTPUT_DIR), path.stat().st_size, "bytes")

## Interpretation

This A100 notebook uses the same official five-fold iPro-MP ensemble as the benchmark. It does not retrain iPro-MP; the A100 change is a larger inference batch for speed while preserving the same model and threshold policy.
There are no reduced epochs because iPro-MP is inference-only. The A100 runtime profile uses sequential fold loading, batch size thirty-two, and omitting train-split
inference by default. Validation and test probabilities remain directly usable
for the shared threshold and held-out comparison policy.